# One Row per Spectrum: Reshaping for Analysis
### 2.8 — Reshaping Data for Analysis

Chemometric and machine-learning tools all want the same shape: **one row per
sample, one column per wavelength** — a samples-by-variables matrix. But data often
arrives *long*: one row per measured point, with sample, wavelength, and absorbance
in separate columns. Getting from one layout to the other is a routine, and
reversible, reshape.

> **One idea to hold onto:** **long** is one row per point (tidy for storage);
> **wide** is one row per spectrum (ready for modelling). `pivot` goes long → wide;
> `melt` goes back.

**By the end of this notebook you will be able to:**

1. Recognize long vs wide layouts.
2. Pivot a long table into a samples-by-wavelengths matrix.
3. Melt it back, and hand the matrix to a model.

## 1. Long (tidy) format

Three samples, each measured at four wavelengths — stored long: every row is a
single (sample, wavelength, absorbance) point. This is how a database or a tidy
export usually gives it to you.

In [1]:
import pandas as pd

long = pd.DataFrame({
    "sample_id":    ["S-01"] * 4 + ["S-02"] * 4 + ["S-03"] * 4,
    "wavelength_nm": [500, 520, 540, 560] * 3,
    "absorbance_AU": [0.10, 0.32, 0.71, 0.40,
                      0.12, 0.35, 0.75, 0.43,
                      0.09, 0.30, 0.68, 0.38],
})
print("long shape:", long.shape, "(one row per measured point)")
long

long shape: (12, 3) (one row per measured point)


,sample_id,wavelength_nm,absorbance_AU
0,S-01,500,0.10
1,S-01,520,0.32
2,S-01,540,0.71
3,S-01,560,0.40
4,S-02,500,0.12
5,S-02,520,0.35
6,S-02,540,0.75
7,S-02,560,0.43
8,S-03,500,0.09
9,S-03,520,0.30


## 2. Why wide?

A model needs each sample as one row and each wavelength as one feature column. In
long form, a single spectrum is spread across several rows — a model can't read that.
The wide layout lines samples up as rows and wavelengths as columns.

## 3. Pivot: long → wide

`pivot` places the unique `sample_id` values down the rows, the `wavelength_nm`
values across the columns, and the absorbances in the cells.

In [2]:
wide = long.pivot(index="sample_id", columns="wavelength_nm", values="absorbance_AU")
print("wide shape:", wide.shape, "(rows = samples, columns = wavelengths)")
wide

wide shape: (3, 4) (rows = samples, columns = wavelengths)


wavelength_nm,500,520,540,560
sample_id,,,,
S-01,0.10,0.32,0.71,0.40
S-02,0.12,0.35,0.75,0.43
S-03,0.09,0.30,0.68,0.38


Three rows, four columns — exactly the samples-by-wavelengths matrix. Each row is now
one complete spectrum.

## 4. Melt: wide → long

`melt` reverses the pivot, collapsing the wavelength columns back into rows. Useful
for storage, for tidy plotting, or to undo a reshape.

In [3]:
back_to_long = wide.reset_index().melt(
    id_vars="sample_id",
    var_name="wavelength_nm",
    value_name="absorbance_AU",
)
print("melted shape:", back_to_long.shape, "(back to one row per point)")
back_to_long.head()

melted shape: (12, 3) (back to one row per point)


,sample_id,wavelength_nm,absorbance_AU
0,S-01,500,0.10
1,S-02,500,0.12
2,S-03,500,0.09
3,S-01,520,0.32
4,S-02,520,0.35


## 5. Hand the matrix to a model

Models work on plain numeric arrays. `.to_numpy()` gives the values; keep the row and
column labels so you know which row is which sample and which column is which wavelength.

In [4]:
X = wide.to_numpy()                 # the numeric samples-by-wavelengths matrix
sample_ids = wide.index.to_numpy()  # which sample each row is
wavelengths = wide.columns.to_numpy()  # which wavelength each column is

print("X shape:", X.shape, "-> (n_samples, n_wavelengths)")
print("rows are samples   :", sample_ids)
print("columns are wavelengths:", wavelengths, "nm")

X shape: (3, 4) -> (n_samples, n_wavelengths)
rows are samples   : ['S-01' 'S-02' 'S-03']
columns are wavelengths: [500 520 540 560] nm


## Key Takeaways

- **Long** = one row per point (good for storage); **wide** = one row per spectrum
  (what models expect).
- `pivot` reshapes long → wide; `melt` reshapes wide → long. They're inverses.
- A wide table's `.to_numpy()` is the samples-by-wavelengths matrix every chemometric
  tool consumes.
- Keep the row index (sample IDs) and columns (wavelengths) so the matrix stays labelled.

## Practical Checklist

- [ ] Know whether your data is long or wide before reshaping.
- [ ] Pivot to one-row-per-sample before modelling.
- [ ] Keep sample IDs and wavelengths as the index/columns, not lost in the reshape.
- [ ] Check the resulting shape is `(n_samples, n_wavelengths)`.

## Common Mistakes

- Feeding a long table to a model that expects one row per sample.
- Pivoting with duplicate (sample, wavelength) pairs, which collide and error.
- Losing track of which axis is samples and which is wavelengths after `.to_numpy()`.

## Next Lesson

You've completed the core of **Track 2 — Scientific Computing**: array math,
region selection, sample tables, missing values, joins, and reshaping. Next is
**Track 3 — Signal Processing**, starting with **3.1 — Noise, Signal, and What
Smoothing Really Does**, where this clean, well-shaped data finally meets
preprocessing.